In [1]:
from numba import njit
import time
import numpy as np
from scipy.spatial.transform import Rotation as R, Slerp

In [2]:
def my_loop():
    i = 0
    s = 0
    while i < 10000000:
        s += i
        i += 1
    return s

@njit
def my_loop_njit():
    i = 0
    s = 0
    while i < 10000000:
        s += i
        i += 1
    return s


In [3]:
for _ in range(10):
    start_time = time.perf_counter()
    result = my_loop()
    end_time = time.perf_counter()
    time_1 = end_time - start_time
    print(f"ohne njit: Ergebnis: {result}  Berechnungszeit: {time_1*1000} Millisekunden")

    start_time = time.perf_counter()
    result = my_loop_njit() 
    end_time = time.perf_counter()
    time_2 = end_time - start_time
    print(f"mit njit: Ergebnis: {result}  Berechnungszeit: {time_2*1000} Millisekunden")

    print(f"mit njit: {time_1/time_2} mal schneller\n")

ohne njit: Ergebnis: 49999995000000  Berechnungszeit: 537.2668000054546 Millisekunden
mit njit: Ergebnis: 49999995000000  Berechnungszeit: 215.6976999831386 Millisekunden
mit njit: 2.4908323085849013 mal schneller

ohne njit: Ergebnis: 49999995000000  Berechnungszeit: 546.5954000246711 Millisekunden
mit njit: Ergebnis: 49999995000000  Berechnungszeit: 0.001900014467537403 Millisekunden
mit njit: 287679.59895227005 mal schneller

ohne njit: Ergebnis: 49999995000000  Berechnungszeit: 533.1773000070825 Millisekunden
mit njit: Ergebnis: 49999995000000  Berechnungszeit: 0.002500019036233425 Millisekunden
mit njit: 213269.2960651921 mal schneller

ohne njit: Ergebnis: 49999995000000  Berechnungszeit: 537.2561999829486 Millisekunden
mit njit: Ergebnis: 49999995000000  Berechnungszeit: 0.0022999593056738377 Millisekunden
mit njit: 233593.78518462277 mal schneller

ohne njit: Ergebnis: 49999995000000  Berechnungszeit: 535.9993999591097 Millisekunden
mit njit: Ergebnis: 49999995000000  Berechnun

In [5]:
@njit
def normalize(q):
    return q / np.sqrt(np.sum(q**2))

@njit
def prepare_slerp(q1, q2):
    q1 = normalize(q1)
    q2 = normalize(q2)
    dot = np.dot(q1, q2)
    if dot < 0.0:
        q2 = -q2
        dot = -dot
    theta_0 = np.arccos(dot)
    sin_theta_0 = np.sin(theta_0)
    return q1, q2, dot, theta_0, sin_theta_0

@njit
def evaluate_slerp(q1, q2, dot, theta_0, sin_theta_0, t):
    if dot > 0.9995:
        result = q1 + t * (q2 - q1)
        return normalize(result)
    theta = theta_0 * t
    sin_theta = np.sin(theta)
    s0 = np.sin(theta_0 - theta) / sin_theta_0
    s1 = sin_theta / sin_theta_0
    return s0 * q1 + s1 * q2






In [6]:
# Beispielquaternionen
q1 = np.array([0.0, 0.0, 0.0, 1.0])
q2 = np.array([0.0, 1.0, 0.0, 0.0])







#scipy
key_times = np.array([0, 1]) 
key_rots = R.from_quat([q1, q2]) 
slerp = Slerp(key_times, key_rots)
start_time = time.perf_counter()
for t in np.linspace(0, 1, 100):
    interpolated_rotation = slerp(t).as_quat()
end_time = time.perf_counter()
time_1 = end_time - start_time
print(f"Benötigte Zeit für scipy-Funktion: {time_1*1000} Millisekunden")







#njit
# Vorbereitung (einmalig)
q1_prep, q2_prep, dot, theta_0, sin_theta_0 = prepare_slerp(q1, q2)

start_time = time.perf_counter()
for t in np.linspace(0, 1, 100):
    q_interp = evaluate_slerp(q1_prep, q2_prep, dot, theta_0, sin_theta_0, t)
end_time = time.perf_counter()
time_2 = end_time - start_time
print(f"Benötigte Zeit für njit-Funktion: {time_2*1000} Millisekunden")


print(f"njit ist {time_1/time_2} mal schneller")

Benötigte Zeit für scipy-Funktion: 4.864000016823411 Millisekunden
Benötigte Zeit für njit-Funktion: 204.61150002665818 Millisekunden
njit ist 0.023771879958798484 mal schneller
